In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [2]:
# Importing MNIST Dataset from torchvision
training_set = datasets.MNIST(root='./data', train=True, download=False, transform=transforms.ToTensor())

# Importing Validation Set from torchvision
validation_set = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

# Creating data loaders
training_loader = DataLoader(training_set, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=32, shuffle=False)

# Class labels
classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

# Report split sizes
print(f'Training set: {len(training_set)}')
print(f'Validation set: {len(validation_set)}')

Training set: 60000
Validation set: 10000


In [3]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 1, out_channels = 6, kernel_size = (5,5), stride = 1, padding = 2)
        self.conv2 = nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = (5,5), stride = 1, padding = 0)
        self.conv3 = nn.Conv2d(in_channels = 16, out_channels = 120, kernel_size = (5,5), stride = 1, padding = 0)
        self.avg_pool = nn.AvgPool2d(2,2)
        self.fc1 = nn.Linear(120,84)
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.tanh(x)
        x = self.avg_pool(x)
        x = self.conv2(x)
        x = F.tanh(x)
        x = self.avg_pool(x)
        x = self.conv3(x)
        x = torch.flatten(x, 1)
        x = F.tanh(x)
        x = self.fc1(x)
        x = F.tanh(x)
        x = self.fc2(x)
        return x

model = LeNet()
device ='cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [4]:
# Defining Loss Function (Since it is a multi-class classification, I chose CrossEntropyLoss)
loss_fn = nn.CrossEntropyLoss()
# Defining Optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [6]:
# Setup optimization loop(s)
epochs = 100

### Train time
# Loop through the epochs
for epoch in range(epochs):
    train_loss_total = 0
    test_loss_total = 0
    train_correct_guess_total = 0
    test_correct_guess_total = 0
    # Set the model to train mode (this is the default)
    model.train(True)
    for images, labels in training_loader:
        images = images.to(device)
        labels = labels.to(device)
        # 1. Do the forward pass
        y_pred = model(images)
        predicted_labels = y_pred.argmax(dim=1)
        train_correct_guess_total += (predicted_labels == labels).sum().item()
        # 2. Calculate the loss (how wrong the model is)
        loss = loss_fn(y_pred, labels)
        # 3. Zero the optimizer gradients
        optimizer.zero_grad()
        # 4. Perform backpropagation
        loss.backward()
        # 5. Step the optimizer
        optimizer.step()
        train_loss_total += loss.item() * images.size(0)
    training_accuracy = (train_correct_guess_total / len(training_set)) * 100
    train_loss_total /= len(training_set)
    ### Test time
    # Set the model to eval mode
    model.eval()
    # Turn on inference mode context manager
    with torch.inference_mode():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)
            # 1. Do the forward pass
            test_pred = model(images)
            predicted_labels = test_pred.argmax(dim=1)
            test_correct_guess_total += (predicted_labels == labels).sum().item()
            # 2. Calculate the loss
            test_loss = loss_fn(test_pred, labels)
            test_loss_total += test_loss.item() * images.size(0)
    test_accuracy = (test_correct_guess_total / len(validation_set)) * 100
    test_loss_total /= len(validation_set)

    # Print out what's happening
    print(f"Epoch: {epoch + 1} | Train loss: {train_loss_total:.4f} | Test loss: {test_loss_total:.4f} | Training Accuracy: {training_accuracy:.2f} % | Test Accuracy: {test_accuracy:.2f} % ")

Epoch: 1 | Train loss: 0.0353 | Test loss: 0.0401 | Training Accuracy: 98.98 % | Test Accuracy: 98.77 % 
Epoch: 2 | Train loss: 0.0337 | Test loss: 0.0404 | Training Accuracy: 99.02 % | Test Accuracy: 98.66 % 
Epoch: 3 | Train loss: 0.0323 | Test loss: 0.0380 | Training Accuracy: 99.08 % | Test Accuracy: 98.77 % 
Epoch: 4 | Train loss: 0.0310 | Test loss: 0.0392 | Training Accuracy: 99.10 % | Test Accuracy: 98.76 % 
Epoch: 5 | Train loss: 0.0301 | Test loss: 0.0382 | Training Accuracy: 99.15 % | Test Accuracy: 98.68 % 
Epoch: 6 | Train loss: 0.0288 | Test loss: 0.0364 | Training Accuracy: 99.20 % | Test Accuracy: 98.72 % 
Epoch: 7 | Train loss: 0.0280 | Test loss: 0.0368 | Training Accuracy: 99.22 % | Test Accuracy: 98.79 % 
Epoch: 8 | Train loss: 0.0269 | Test loss: 0.0363 | Training Accuracy: 99.24 % | Test Accuracy: 98.74 % 
Epoch: 9 | Train loss: 0.0260 | Test loss: 0.0367 | Training Accuracy: 99.29 % | Test Accuracy: 98.83 % 
Epoch: 10 | Train loss: 0.0252 | Test loss: 0.0343 | Tr

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'lenet.pth')